In [2]:
"""
Self-Attention Encoder — Step by Step Implementation
======================================================
Reproduces the full worked example from the document:
"OMG!!! AI students rrr brillianttt :)" -> context-aware representations
via a single Transformer encoder block (Self-Attention -> Add&Norm ->
FeedForward -> Add&Norm).

All the numbers (embeddings, weight matrices) are taken directly from
the worked example so you can check every intermediate value.
"""

import numpy as np

np.set_printoptions(precision=4, suppress=True)


#  Raw text & Preprocessing
def preprocess(text: str) -> str:
    """Very small illustrative cleaner: lowercase, drop junk symbols,
    collapse repeated letters, and normalize a couple of known slang words.
    (In a real system this would be a full text-normalization pipeline.)"""
    import re

    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()

    fixes = {
        "rrr": "are",
        "ar": "are",
        "briliant": "brilliant",
        "omg": "",
    }
    tokens = [fixes.get(w, w) for w in text.split()]
    tokens = [t for t in tokens if t]
    return " ".join(tokens)


T0 = "OMG!!! AI students rrr brillianttt :)"
T1 = "ai students are brilliant"
print("STEP 1 — Raw text:      ", T0)
print("STEP 2 — Preprocessed:  ", T1)



STEP 1 — Raw text:       OMG!!! AI students rrr brillianttt :)
STEP 2 — Preprocessed:   ai students are brilliant


In [3]:

# STEP 3-4 — Tokenization & Token IDs
tokens = T1.split()
token_to_id = {"ai": 101, "students": 205, "are": 306, "brilliant": 412}
token_ids = [token_to_id[t] for t in tokens]

print("\nSTEP 3 — Tokens:        ", tokens)
print("STEP 4 — Token IDs:     ", token_ids)




STEP 3 — Tokens:         ['ai', 'students', 'are', 'brilliant']
STEP 4 — Token IDs:      [101, 205, 306, 412]


In [4]:

# STEP 5 — Embedding lookup
d_model = 4
embedding_table = {
    101: [0.20, 0.40, 0.10, 0.50],   # ai
    205: [0.60, 0.10, 0.80, 0.30],   # students
    306: [0.10, 0.70, 0.20, 0.90],   # are
    412: [0.90, 0.30, 0.60, 0.20],   # brilliant
}
X = np.array([embedding_table[i] for i in token_ids])  # shape (4,4)

print("\nSTEP 5 — Embedding matrix X (tokens x d_model):")
print(X)




STEP 5 — Embedding matrix X (tokens x d_model):
[[0.2 0.4 0.1 0.5]
 [0.6 0.1 0.8 0.3]
 [0.1 0.7 0.2 0.9]
 [0.9 0.3 0.6 0.2]]


In [5]:

# STEP 6 — Positional Encoding
def positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (i / d_model))
            pe[pos, i] = np.sin(angle)
            if i + 1 < d_model:
                pe[pos, i + 1] = np.cos(angle)
    return pe


PE = positional_encoding(len(tokens), d_model)
print("\nSTEP 6 — Positional Encoding matrix:")
print(PE)

Z = X + PE
print("\nSTEP 6.3 — Z = X + PE (final transformer input):")
print(Z)




STEP 6 — Positional Encoding matrix:
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]

STEP 6.3 — Z = X + PE (final transformer input):
[[ 0.2     1.4     0.1     1.5   ]
 [ 1.4415  0.6403  0.81    1.3   ]
 [ 1.0093  0.2839  0.22    1.8998]
 [ 1.0411 -0.69    0.63    1.1996]]


In [6]:

# STEP 7 — Create Query, Key, Value matrices
W_Q = np.array([[1, 0],
                [0, 1],
                [1, 0],
                [0, 1]], dtype=float)

W_K = np.array([[1, 0],
                [0, 1],
                [0.5, 0],
                [0, 0.5]], dtype=float)

W_V = np.array([[0.5, 0],
                [0, 1],
                [1, 0],
                [0, 0.5]], dtype=float)

Q = Z @ W_Q
K = Z @ W_K
V = Z @ W_V

print("\nSTEP 7 — Q, K, V matrices:")
print("Q =\n", Q)
print("K =\n", K)
print("V =\n", V)



STEP 7 — Q, K, V matrices:
Q =
 [[0.3    2.9   ]
 [2.2515 1.9403]
 [1.2293 2.1837]
 [1.6711 0.5096]]
K =
 [[ 0.25    2.15  ]
 [ 1.8465  1.2903]
 [ 1.1193  1.2338]
 [ 1.3561 -0.0902]]
V =
 [[ 0.2     2.15  ]
 [ 1.5307  1.2903]
 [ 0.7246  1.2338]
 [ 1.1506 -0.0902]]


In [7]:


# STEP 8 — Query-Key comparison (raw attention scores)
S = Q @ K.T
print("\nSTEP 8 — Raw score matrix S = Q.K^T:")
print(S)




STEP 8 — Raw score matrix S = Q.K^T:
[[6.31   4.2957 3.9137 0.1452]
 [4.7344 6.6607 4.9139 2.8782]
 [5.0022 5.0874 4.07   1.4701]
 [1.5133 3.7431 2.4991 2.2203]]


In [8]:

# STEP 9 — Scale the scores
d_k = K.shape[1]  # 2
S_scaled = S / np.sqrt(d_k)
print(f"\nSTEP 9 — Scaled scores S' = S / sqrt(d_k), d_k={d_k}:")
print(S_scaled)




STEP 9 — Scaled scores S' = S / sqrt(d_k), d_k=2:
[[4.4618 3.0376 2.7674 0.1027]
 [3.3477 4.7099 3.4746 2.0352]
 [3.5371 3.5973 2.878  1.0395]
 [1.0701 2.6468 1.7672 1.57  ]]


In [9]:

# STEP 10 — Softmax -> Attention weights
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)   # numerical stability
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


A = softmax(S_scaled, axis=1)
print("\nSTEP 10 — Attention weights A = softmax(S'):")
print(A)

labels = tokens
print("\nSelf-attention matrix (rows = query word, cols = key word):")
header = "          " + "".join(f"{c:>12}" for c in labels)
print(header)
for i, row_label in enumerate(labels):
    row_vals = "".join(f"{A[i, j]*100:11.1f}%" for j in range(len(labels)))
    print(f"{row_label:>10}{row_vals}")



STEP 10 — Attention weights A = softmax(S'):
[[0.6958 0.1675 0.1278 0.0089]
 [0.1585 0.6189 0.18   0.0427]
 [0.3757 0.399  0.1944 0.0309]
 [0.1053 0.5096 0.2115 0.1736]]

Self-attention matrix (rows = query word, cols = key word):
                    ai    students         are   brilliant
        ai       69.6%       16.7%       12.8%        0.9%
  students       15.9%       61.9%       18.0%        4.3%
       are       37.6%       39.9%       19.4%        3.1%
 brilliant       10.5%       51.0%       21.1%       17.4%


In [10]:


# STEP 11 — Apply weights to values -> attention output
O = A @ V
print("\nSTEP 11 — Attention output O = A.V:")
print(O)




STEP 11 — Attention output O = A.V:
[[0.4984 1.869 ]
 [1.1585 1.3575]
 [0.8624 1.5596]
 [1.1541 1.1292]]


In [11]:


# STEP 12 — Output projection back to d_model
W_O = np.array([[1, 0, 0, 1],
                [0, 1, 1, 0]], dtype=float)

H = O @ W_O
print("\nSTEP 12 — Output projection H = O.W_O (back to d_model=4):")
print(H)




STEP 12 — Output projection H = O.W_O (back to d_model=4):
[[0.4984 1.869  1.869  0.4984]
 [1.1585 1.3575 1.3575 1.1585]
 [0.8624 1.5596 1.5596 0.8624]
 [1.1541 1.1292 1.1292 1.1541]]


In [12]:

# STEP 13 — Residual connection
R = Z + H
print("\nSTEP 13 — Residual R = Z + H:")
print(R)




STEP 13 — Residual R = Z + H:
[[0.6984 3.269  1.969  1.9984]
 [2.6    1.9978 2.1675 2.4585]
 [1.8717 1.8435 1.7796 2.7622]
 [2.1953 0.4392 1.7592 2.3537]]


In [13]:

# STEP 14 — Layer Normalization
def layer_norm(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)


Y = layer_norm(R)
print("\nSTEP 14 — LayerNorm(R) -> Y (attention sub-layer output):")
print(Y)




STEP 14 — LayerNorm(R) -> Y (attention sub-layer output):
[[-1.4141  1.4141 -0.0162  0.0162]
 [ 1.243  -1.3025 -0.5852  0.6448]
 [-0.4763 -0.546  -0.7039  1.7261]
 [ 0.6756 -1.658   0.0962  0.8862]]


In [14]:

# STEP 15 — Feed-Forward Network (position-wise)
def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


W1 = np.array([[1, 0, 1],
               [0, 1, 1],
               [1, 1, 0],
               [1, -1, 1]], dtype=float)   # (4x3)

W2 = np.array([[1, 0, 1, 0],
               [0, 1, 0, 1],
               [1, 1, 0, 1]], dtype=float)  # (3x4)

b1 = np.zeros(3)
b2 = np.zeros(4)


def feed_forward(y: np.ndarray) -> np.ndarray:
    hidden = gelu(y @ W1 + b1)
    return hidden @ W2 + b2


FFN_Y = feed_forward(Y)
print("\nSTEP 15 — FFN(Y) = GELU(Y.W1 + b1).W2 + b2:")
print(FFN_Y)



STEP 15 — FFN(Y) = GELU(Y.W1 + b1).W2 + b2:
[[-0.1033  1.2743 -0.1115  1.2743]
 [ 1.5986  0.4079  1.1768  0.4079]
 [ 0.9206  0.5305  0.3862  0.5305]
 [ 1.5328 -0.0616  1.5772 -0.0616]]


In [15]:


# STEP 16 — Second residual connection
R2 = Y + FFN_Y
print("\nSTEP 16 — Second residual R2 = Y + FFN(Y):")
print(R2)




STEP 16 — Second residual R2 = Y + FFN(Y):
[[-1.5174  2.6884 -0.1276  1.2905]
 [ 2.8416 -0.8946  0.5916  1.0527]
 [ 0.4444 -0.0155 -0.3177  2.2566]
 [ 2.2084 -1.7195  1.6733  0.8246]]


In [16]:

# STEP 17 — Second Layer Normalization -> final encoder output
Z_out = layer_norm(R2)
print("\nSTEP 17 — Z_out = LayerNorm(R2)  <-- Final context-aware encoding")
print(Z_out)

print("\nFinal context-aware representations per word:")
for word, vec in zip(labels, Z_out):
    print(f"  {word:>10}: {vec}")




STEP 17 — Z_out = LayerNorm(R2)  <-- Final context-aware encoding
[[-1.3388  1.3414 -0.4532  0.4505]
 [ 1.4581 -1.3445 -0.2297  0.1162]
 [-0.1478 -0.6083 -0.9108  1.6669]
 [ 0.97   -1.6366  0.6149  0.0517]]

Final context-aware representations per word:
          ai: [-1.3388  1.3414 -0.4532  0.4505]
    students: [ 1.4581 -1.3445 -0.2297  0.1162]
         are: [-0.1478 -0.6083 -0.9108  1.6669]
   brilliant: [ 0.97   -1.6366  0.6149  0.0517]


In [17]:

# BONUS — a from-scratch reusable self-attention function
# (this bundles steps 7-11 into a single callable, matching the formula
#  Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V )
def self_attention(Z, W_Q, W_K, W_V):
    Q = Z @ W_Q
    K = Z @ W_K
    V = Z @ W_V
    d_k = K.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights


if __name__ == "__main__":
    out, attn_weights = self_attention(Z, W_Q, W_K, W_V)
    print("\n[Sanity check] self_attention() reproduces STEP 11 output:")
    print(out)
    assert np.allclose(out, O), "Mismatch with step-by-step O!"
    print("OK — matches manual step-by-step calculation.")


[Sanity check] self_attention() reproduces STEP 11 output:
[[0.4984 1.869 ]
 [1.1585 1.3575]
 [0.8624 1.5596]
 [1.1541 1.1292]]
OK — matches manual step-by-step calculation.
